# 08 — Modélisation : Random Forest

**Objectif** : entraîner et optimiser une forêt aléatoire pour la détection d'anomalies CBC, sur les deux configurations de features (Expérience A : 9 features, Expérience B1 : 5 features).

**Principe** : Random Forest combine de nombreux Decision Trees entraînés sur des échantillons bootstrap différents (bagging) et avec un sous-ensemble aléatoire de variables à chaque division (feature randomness), puis agrège leurs prédictions par vote majoritaire. Cela réduit la variance par rapport à un arbre unique (étape 7), tout en conservant sa capacité à capturer des règles par seuils.

**Hypothèse à vérifier** : Random Forest devrait au moins égaler le Decision Tree optimisé, avec une meilleure généralisation attendue sur l'Expérience B1.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import time
import joblib
import os
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from evaluation import evaluate_model, plot_confusion_matrices, results_table

os.makedirs('../models', exist_ok=True)
os.makedirs('../figures', exist_ok=True)

RANDOM_STATE = 42

## 0. Chargement des données préparées (issues de l'étape 4)

In [ ]:
X_train_A = pd.read_csv('../data/processed/X_train_A.csv')
X_test_A  = pd.read_csv('../data/processed/X_test_A.csv')
y_train_A = pd.read_csv('../data/processed/y_train_A.csv').squeeze()
y_test_A  = pd.read_csv('../data/processed/y_test_A.csv').squeeze()

X_train_B1 = pd.read_csv('../data/processed/X_train_B1.csv')
X_test_B1  = pd.read_csv('../data/processed/X_test_B1.csv')
y_train_B1 = pd.read_csv('../data/processed/y_train_B1.csv').squeeze()
y_test_B1  = pd.read_csv('../data/processed/y_test_B1.csv').squeeze()

print("Expérience A :", X_train_A.shape, X_test_A.shape)
print("Expérience B1:", X_train_B1.shape, X_test_B1.shape)

## 1. Baseline — Random Forest avec paramètres par défaut (100 arbres)

In [ ]:
rf_A_baseline = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
model_A_base, y_pred_A_base, metrics_A_base = evaluate_model(
    rf_A_baseline, X_train_A, y_train_A, X_test_A, y_test_A,
    "RF baseline - Exp A"
)

rf_B1_baseline = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
model_B1_base, y_pred_B1_base, metrics_B1_base = evaluate_model(
    rf_B1_baseline, X_train_B1, y_train_B1, X_test_B1, y_test_B1,
    "RF baseline - Exp B1"
)

results_table([metrics_A_base, metrics_B1_base])

## 2. Recherche d'hyperparamètres — GridSearchCV (Expérience A)

Grille testée :
- `n_estimators` : nombre d'arbres dans la forêt
- `max_depth` : profondeur max de chaque arbre
- `min_samples_split` / `min_samples_leaf` : régularisation
- `max_features` : nombre de variables considérées à chaque division

⚠️ Note : la grille est volontairement plus compacte que pour Decision Tree, car chaque combinaison entraîne `n_estimators` arbres au lieu d'un seul — le temps de calcul total augmente vite.

In [ ]:
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 5],
    'max_features': ['sqrt', 'log2']
}

grid_rf_A = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=param_grid_rf,
    scoring='f1',
    cv=3,                 # cv réduit à 3 (au lieu de 5) pour limiter le temps de calcul
    n_jobs=-1,
    verbose=1
)

start = time.time()
grid_rf_A.fit(X_train_A, y_train_A)
print(f"Temps de recherche: {time.time()-start:.1f}s")

print("\nMeilleurs paramètres (Exp A):", grid_rf_A.best_params_)
print("Meilleur F1-score (CV, Exp A):", grid_rf_A.best_score_)

## 3. Recherche d'hyperparamètres — GridSearchCV (Expérience B1)

In [ ]:
grid_rf_B1 = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=param_grid_rf,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    verbose=1
)

start = time.time()
grid_rf_B1.fit(X_train_B1, y_train_B1)
print(f"Temps de recherche: {time.time()-start:.1f}s")

print("\nMeilleurs paramètres (Exp B1):", grid_rf_B1.best_params_)
print("Meilleur F1-score (CV, Exp B1):", grid_rf_B1.best_score_)

## 4. Évaluation finale des meilleurs modèles sur le TEST set

In [ ]:
best_rf_A = grid_rf_A.best_estimator_
best_rf_B1 = grid_rf_B1.best_estimator_

y_pred_A = best_rf_A.predict(X_test_A)
y_pred_B1 = best_rf_B1.predict(X_test_B1)

results = []
for name, y_true, y_pred in [
    ("RF baseline - Exp A", y_test_A, y_pred_A_base),
    ("RF optimisé - Exp A", y_test_A, y_pred_A),
    ("RF baseline - Exp B1", y_test_B1, y_pred_B1_base),
    ("RF optimisé - Exp B1", y_test_B1, y_pred_B1),
]:
    results.append({
        'Modèle': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-score': f1_score(y_true, y_pred),
    })

df_results = results_table(results)
df_results

## 5. Matrices de confusion (baseline vs optimisé, A vs B1)

In [ ]:
configs = [
    (y_test_A, y_pred_A_base, "RF baseline - Exp A"),
    (y_test_A, y_pred_A, "RF optimisé - Exp A"),
    (y_test_B1, y_pred_B1_base, "RF baseline - Exp B1"),
    (y_test_B1, y_pred_B1, "RF optimisé - Exp B1"),
]

plot_confusion_matrices(configs, save_path='../figures/confusion_matrices_rf.png')

## 6. Importance des variables (moyennée sur tous les arbres de la forêt)

Plus stable/fiable que celle d'un Decision Tree unique, car moyennée sur des centaines d'arbres entraînés sur des sous-échantillons différents.

In [ ]:
importance_A = pd.DataFrame({
    'Variable': X_train_A.columns,
    'Importance': best_rf_A.feature_importances_
}).sort_values('Importance', ascending=False)

importance_B1 = pd.DataFrame({
    'Variable': X_train_B1.columns,
    'Importance': best_rf_B1.feature_importances_
}).sort_values('Importance', ascending=False)

print("Importance des variables (Expérience A):")
print(importance_A)
print("\nImportance des variables (Expérience B1):")
print(importance_B1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].barh(importance_A['Variable'], importance_A['Importance'], color='steelblue')
axes[0].set_title('Feature importance - Expérience A')
axes[0].invert_yaxis()

axes[1].barh(importance_B1['Variable'], importance_B1['Importance'], color='salmon')
axes[1].set_title('Feature importance - Expérience B1')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../figures/feature_importance_rf.png', dpi=150)
plt.show()

## 7. Comparaison directe Decision Tree vs Random Forest

On recharge les résultats du Decision Tree (étape 7) pour comparer directement l'apport du Random Forest.

In [ ]:
try:
    df_results_dt = pd.read_csv('../data/processed/results_dt.csv')
    comparison = pd.concat([
        df_results_dt[df_results_dt['Modèle'].str.contains('optimisé')],
        df_results[df_results['Modèle'].str.contains('optimisé')]
    ], ignore_index=True)
    print("Comparaison Decision Tree (optimisé) vs Random Forest (optimisé):")
    comparison
except FileNotFoundError:
    print("⚠️ results_dt.csv non trouvé — exécutez d'abord le notebook 07 (Decision Tree)")

## 8. Sauvegarde des modèles et résultats

In [ ]:
joblib.dump(best_rf_A, '../models/rf_A_best.pkl')
joblib.dump(best_rf_B1, '../models/rf_B1_best.pkl')

df_results.to_csv('../data/processed/results_rf.csv', index=False)
print("✅ Modèles et résultats Random Forest sauvegardés")

## Ce qu'il faut analyser pour le rapport

1. **RF vs Decision Tree** : Random Forest a-t-il amélioré le F1-score par rapport au Decision Tree seul, en particulier sur l'Expérience B1 (où la réduction de variance devrait le plus aider) ?
2. **`n_estimators` retenu** : 100 ou 200 arbres ont-ils fait une différence notable, ou les gains sont-ils marginaux (rendements décroissants) ?
3. **Feature importance** : cohérence avec le Decision Tree et avec la pondération médicale de l'étape 3 — Random Forest donne généralement une estimation plus stable.
4. **Temps de calcul** : noter l'écart de temps d'entraînement avec Decision Tree (RF est plus lent, normal puisqu'il entraîne des centaines d'arbres) — un compromis performance/coût à mentionner dans la conclusion.